# NovaCrédit S.A.S. — Sesión 2: Pandas y datos reales
## Módulo 3 · Diplomado IA y Analítica de Datos · UMNG

---

**Analista:** [Tu nombre completo]  
**Fecha:** 28 de junio de 2025  
**Proyecto:** NovaCrédit — Pipeline de analítica  
**Versión:** 1.0

---

### Contexto

El Director Comercial necesita un informe con **6 KPIs** para la reunión del lunes.  
Los archivos de datos acaban de llegar. Tu trabajo: convertirlos en información útil.

```
Cargar → Diagnosticar → Limpiar → Calcular KPIs → Conclusión ejecutiva
```

## 0. Configuración inicial

Primero instalamos / importamos las librerías y cargamos los archivos desde GitHub.

In [ ]:
# ── Librerías ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

print("pandas", pd.__version__)
print("numpy", np.__version__)

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────────
# Reemplaza las URLs por los enlaces reales de tu Google Drive / GitHub
BASE = "https://raw.githubusercontent.com/TU_USUARIO/novacredit/main/datos/"

clientes = pd.read_csv(BASE + "clientes.csv",
                       parse_dates=["fecha_vinculacion"])

creditos = pd.read_csv(BASE + "creditos.csv",
                       parse_dates=["fecha_desembolso"])

campanas = pd.read_csv(BASE + "campanas.csv",
                       parse_dates=["fecha_inicio", "fecha_fin"])

print("Archivos cargados")
print(f"  clientes : {clientes.shape}")
print(f"  creditos : {creditos.shape}")
print(f"  campanas : {campanas.shape}")

---
## Bloque 1 — Diagnóstico de calidad

> **Regla del analista:** antes de calcular cualquier KPI,  
> documenta el estado real de los datos. Nunca trabajes a ciegas.

In [ ]:
# ── 1.1 Exploración rápida ────────────────────────────────────────────────────
# Tip: .head() muestra las primeras 5 filas
clientes.head()

In [ ]:
# Dimensiones y tipos de dato
print("=== CLIENTES ===")
print(f"Filas: {clientes.shape[0]}  |  Columnas: {clientes.shape[1]}")
print()
print(clientes.dtypes)

In [ ]:
# ── 1.2 Nulos ─────────────────────────────────────────────────────────────────
# isnull().sum() cuenta cuántos NaN hay por columna
print("=== Nulos por columna ===")
nulos_cli = clientes.isnull().sum()
print(nulos_cli[nulos_cli > 0])   # Solo columnas con nulos
print()

print("=== Nulos en CRÉDITOS ===")
nulos_cred = creditos.isnull().sum()
print(nulos_cred[nulos_cred > 0])

In [ ]:
# ── 1.3 Duplicados ────────────────────────────────────────────────────────────
dup_cli = clientes.duplicated().sum()
dup_cred = creditos.duplicated().sum()

print(f"Duplicados en clientes : {dup_cli}")
print(f"Duplicados en créditos : {dup_cred}")

In [ ]:
# ── 1.4 Estadísticas descriptivas ────────────────────────────────────────────
# Tip: .describe() resume columnas numéricas
creditos[["monto", "plazo_meses", "tasa_ea", "dias_mora"]].describe().round(2)

### Tu diagnóstico

Completa el siguiente resumen antes de seguir:

| Problema | Tabla | Columna | Registros afectados |
|----------|-------|---------|---------------------|
| Nulos    | clientes | ciudad | ? |
| Nulos    | clientes | ingreso_mensual | ? |
| Nulos    | creditos | monto | ? |
| Duplicados | clientes | (toda la fila) | ? |

---
## Bloque 2 — Limpieza de datos

Ahora que conocemos los problemas, los corregimos **antes** de calcular cualquier KPI.

In [ ]:
# ── 2.1 Eliminar duplicados ───────────────────────────────────────────────────
# drop_duplicates() elimina filas idénticas
filas_antes = len(clientes)
clientes = clientes.drop_duplicates()
filas_despues = len(clientes)

print(f"Duplicados eliminados: {filas_antes - filas_despues}")
print(f"Clientes únicos: {filas_despues}")

In [ ]:
# ── 2.2 Rellenar nulos con fillna() ──────────────────────────────────────────
# Ciudad sin dato → "Sin ciudad"
clientes["ciudad"] = clientes["ciudad"].fillna("Sin ciudad")

# Ingreso sin dato → mediana del grupo (más justo que el promedio)
mediana_ingreso = clientes["ingreso_mensual"].median()
clientes["ingreso_mensual"].fillna(mediana_ingreso, inplace=True)

print("Nulos restantes en clientes:")
print(clientes.isnull().sum()[clientes.isnull().sum() > 0])
print("→ Sin nulos" if clientes.isnull().sum().sum() == 0 else "")

In [ ]:
# ── 2.3 Eliminar filas con nulos críticos en créditos ────────────────────────
# Si no hay monto, el crédito no tiene sentido para el análisis
cred_antes = len(creditos)
creditos = creditos.dropna(subset=["monto"])
cred_despues = len(creditos)

print(f"Créditos eliminados (sin monto): {cred_antes - cred_despues}")
print(f"Créditos válidos: {cred_despues}")

In [ ]:
# ── 2.4 Verificar tipos de dato ───────────────────────────────────────────────
# La fecha_desembolso debe ser datetime, no object
print(creditos["fecha_desembolso"].dtype)

# Si aparece 'object', lo convertimos:
# creditos["fecha_desembolso"] = pd.to_datetime(creditos["fecha_desembolso"])

# Agregar columna de año para usar después
creditos["anio_desembolso"] = creditos["fecha_desembolso"].dt.year
print(creditos["anio_desembolso"].value_counts().sort_index())

---
## Bloque 3 — KPI 1 y KPI 2

**KPI 1:** Cartera total desembolsada  
**KPI 2:** Ticket promedio por producto

In [ ]:
# ── KPI 1: Cartera total ──────────────────────────────────────────────────────
# Solo créditos activos (no cancelados ni castigados)
cartera_activa = creditos[creditos["estado"].isin(["Al día", "En mora"])]

cartera_total = cartera_activa["monto"].sum()

print(f"KPI 1 — Cartera total desembolsada activa:")
print(f"  ${cartera_total:,.0f} COP")
print(f"  ${cartera_total/1_000_000:.1f} millones de pesos")

In [ ]:
# ── KPI 2: Ticket promedio por producto ───────────────────────────────────────
# groupby agrupa las filas por producto y .mean() calcula el promedio de monto
ticket_promedio = (cartera_activa
                   .groupby("producto")["monto"]
                   .mean()
                   .sort_values(ascending=False)
                   .round(0))

print("KPI 2 — Ticket promedio por producto (COP):")
print(ticket_promedio.apply(lambda x: f"  ${x:,.0f}"))
print()
print(f"Producto con mayor ticket: {ticket_promedio.idxmax()}")

---
## Bloque 4 — KPI 3, KPI 4 y KPI 5

**KPI 3:** Tasa de mora  
**KPI 4:** Canal más rentable  
**KPI 5:** Clientes nuevos vs recurrentes

In [ ]:
# ── KPI 3: Tasa de mora ───────────────────────────────────────────────────────
# Un cliente está en mora si tiene dias_mora > 0
total_creditos_activos = len(cartera_activa)
en_mora = (cartera_activa["dias_mora"] > 0).sum()

tasa_mora = en_mora / total_creditos_activos * 100

print(f"KPI 3 — Tasa de mora:")
print(f"  Créditos en mora   : {en_mora:,}")
print(f"  Total activos      : {total_creditos_activos:,}")
print(f"  Tasa de mora       : {tasa_mora:.1f}%")

# Distribucion por rango de mora
bins = [0, 30, 60, 90, 180, float('inf')]
labels = ["1-30 días", "31-60 días", "61-90 días", "91-180 días", ">180 días"]
en_mora_df = cartera_activa[cartera_activa["dias_mora"] > 0].copy()
en_mora_df["rango_mora"] = pd.cut(en_mora_df["dias_mora"], bins=bins, labels=labels)
print()
print(en_mora_df["rango_mora"].value_counts().sort_index())

In [ ]:
# ── KPI 4: Canal más rentable ─────────────────────────────────────────────────
# Rentabilidad = monto total desembolsado por canal
rentabilidad_canal = (cartera_activa
                      .groupby("canal")["monto"]
                      .agg(total="sum", num_creditos="count", promedio="mean")
                      .sort_values("total", ascending=False)
                      .round(0))

print("KPI 4 — Rentabilidad por canal:")
print(rentabilidad_canal)
print()
print(f"Canal más rentable: {rentabilidad_canal.index[0]}")

In [ ]:
# ── KPI 5: Clientes nuevos vs recurrentes ─────────────────────────────────────
# Un cliente es recurrente si tiene más de 1 crédito en el sistema
creditos_por_cliente = (creditos
                        .groupby("id_cliente")["id_credito"]
                        .count()
                        .reset_index()
                        .rename(columns={"id_credito": "num_creditos"}))

nuevos     = (creditos_por_cliente["num_creditos"] == 1).sum()
recurrentes = (creditos_por_cliente["num_creditos"] > 1).sum()
total_con_credito = len(creditos_por_cliente)

print("KPI 5 — Clientes nuevos vs recurrentes:")
print(f"  Nuevos      (1 crédito)  : {nuevos:,}  ({nuevos/total_con_credito*100:.1f}%)")
print(f"  Recurrentes (2+ créditos): {recurrentes:,}  ({recurrentes/total_con_credito*100:.1f}%)")
print()

# Top clientes por número de créditos
print("Top 5 clientes más activos:")
print(creditos_por_cliente.sort_values("num_creditos", ascending=False).head())

---
## Bloque 5 — KPI 6: Eficiencia de campañas

**KPI 6:** Costo por cliente adquirido por campaña  

Para este KPI necesitamos cruzar la tabla `campanas` con los resultados.

In [ ]:
# ── Preparar campanas ─────────────────────────────────────────────────────────
# Solo campanas finalizadas con conversiones > 0
camp_finalizadas = campanas[
    (campanas["estado"] == "Finalizada") &
    (campanas["conversiones"] > 0)
].copy()

# Eliminar nulos en costo_total
camp_finalizadas = camp_finalizadas.dropna(subset=["costo_total"])

print(f"Campanas finalizadas con datos completos: {len(camp_finalizadas)}")

In [ ]:
# ── KPI 6: Costo por cliente adquirido (CPA) ──────────────────────────────────
camp_finalizadas["cpa"] = (camp_finalizadas["costo_total"] /
                            camp_finalizadas["conversiones"]).round(0)

camp_finalizadas["tasa_conversion"] = (
    camp_finalizadas["conversiones"] /
    camp_finalizadas["clientes_contactados"] * 100
).round(1)

# Resumen por tipo de campaña
resumen_tipo = (camp_finalizadas
                .groupby("tipo")
                .agg(
                    campanas=("id_campana", "count"),
                    cpa_promedio=("cpa", "mean"),
                    tasa_conv_promedio=("tasa_conversion", "mean")
                )
                .round(1)
                .sort_values("cpa_promedio"))

print("KPI 6 — Eficiencia por tipo de campaña (CPA = costo por adquisición):")
print(resumen_tipo)
print()
print(f"Canal más eficiente (menor CPA): {resumen_tipo.index[0]}")

In [ ]:
# Top 5 campanas más eficientes
print("Top 5 campanas con menor costo por adquisición:")
top5 = camp_finalizadas.nsmallest(5, "cpa")[["nombre_campana", "tipo", "conversiones", "cpa"]]
print(top5.to_string(index=False))

---
## Bloque 6 — Conclusión ejecutiva para el Director Comercial

El siguiente texto es tu entrega final. Completa los valores con los resultados que calculaste.

In [ ]:
# ── Resumen automático de KPIs ────────────────────────────────────────────────
print("="*60)
print("INFORME EJECUTIVO — NovaCrédit S.A.S.")
print("Director Comercial · Preparado por el equipo de datos")
print("="*60)
print()
print(f"KPI 1 — Cartera activa total  : ${cartera_total/1e6:.1f} M COP")
print(f"KPI 2 — Ticket promedio más alto: {ticket_promedio.idxmax()} ${ticket_promedio.max()/1e6:.1f} M")
print(f"KPI 3 — Tasa de mora          : {tasa_mora:.1f}%")
print(f"KPI 4 — Canal más rentable    : {rentabilidad_canal.index[0]}")
print(f"KPI 5 — Clientes recurrentes  : {recurrentes/total_con_credito*100:.1f}%")
print(f"KPI 6 — Canal más eficiente   : {resumen_tipo.index[0]}")
print()
print("="*60)

### Interpretación ejecutiva

Escribe aquí **3 conclusiones de negocio** basadas en los números:

1. **[Tu conclusión sobre la cartera y mora]**

2. **[Tu conclusión sobre canales]**

3. **[Tu recomendación al Director Comercial]**

---
*Notebook generado como parte del Módulo 3 — Diplomado IA y Analítica · UMNG · 2025*